# SimMAP Data Exploration

Notebook para visualizar os dados da API do projeto e da tabela no Notion.


In [22]:
from dotenv import load_dotenv
from utils.sanitizer import (
    tratar_dataframe_notion,
    sanitize_string,
    build_nome_lookup,
    build_project_series,
    convert_comma_strings_to_lists,
    standardize_project_keys,
    split_participantes,
    normalizar_participante,
    merge_dataframe_columns_values,
    replace_nan_with_none
)
import os
import json
import requests
import pandas as pd

In [23]:
# Leitura do dotenv
load_dotenv()

sheet_api_url = os.getenv("SHEETS_API_URL")

## Leitura dos dados
As fontes de dados são as seguintes:
- Planilha excel com dados coletados do forms.
- Notion geral da LAWD
- Notion da Diretoria de Projetos da LAWD

In [24]:
# Leitura dos dados da planilha do forms
if not sheet_api_url:
    raise ValueError("A variável SHEETS_API_URL não foi definida no .env")

response = requests.get(sheet_api_url, timeout=30)
response.raise_for_status()
data = response.json()

df_map = pd.DataFrame(data["mapeamento"])

df_map = df_map.drop(columns=["id"])

df_map.head(3)

,nome,curso,periodo,disponibilidade,tech,carreira,maturidade,projetosAtuais,projetosFuturos
0,Bruno Saint Clair Silva Oliveira,SI (Sistemas de Informação),8,De 6 a 8 horas semanais,"Java, JavaScript, TypeScript, Python, HTML e C...",Desenvolvedor Back-end,Autônomo (Resolve problemas sozinho),SIMGrade,"Site da LAWD, POA - MPSE"
1,Álvaro Luís Silva Peixoto,CC (Ciência da Computação),10,Menos de 6 horas semanais,"PHP, Javascript, Golang, Typescript, C#, Ban...","Desenvolvedor Back-end, Full Stack",Autônomo (Resolve problemas sozinho),Nenhum,"ufs.br, Site da LAWD, POA - MPSE, TransformAção"
2,André Felipe de Santana Conceição,SI (Sistemas de Informação),7,De 6 a 8 horas semanais,"HTML, CSS, JS, React, NodeJs, NextJS, Git, Git...",Full Stack,Em desenvolvimento (Já executa tarefas com apoio),Nenhum,"ufs.br, Site da LAWD, SIMCalc, POA - MPSE, Tra..."


In [25]:
# Leitura dos CSVs do Notion Geral da LAWD
notion_dir = os.path.join("data", "notion-lawd")

df_notion1 = pd.read_csv(
    os.path.join(notion_dir, "Membros b8415a3d3d19462690637c8ea91a09e3_all.csv")
)

df_notion2 = pd.read_csv(
    os.path.join(notion_dir, "Membros b8415a3d3d19462690637c8ea91a09e3.csv")
)

# Fazendo o merge dos dois dataframes
df_notion = pd.merge(df_notion1, df_notion2, on="Name")

df_notion.head(3)

,Name,Aniversário_x,Arquivos e mídia_x,Cadastro_x,Email_x,Entrada_x,Nome Completo_x,Phone_x,Projetos_x,Resumo da IA_x,...,Phone_y,Aniversário_y,Áreas de interesse_y,Techs_y,Cadastro_y,Arquivos e mídia_y,Projetos_y,Resumo da IA_y,Seleção_y,Nome Completo_y
0,Gyovani Santos,19 de março de 2025,NaN,incompleto,gyovani.santos@dcomp.ufs.br,NaN,Gyovani Yuri Souza Santos,79988697393,WebGarden (https://www.notion.so/WebGarden-250...,NaN,...,79988697393,19 de março de 2025,"Desenvolvimento web, Redes","CAD, Redes",incompleto,NaN,WebGarden (https://www.notion.so/WebGarden-250...,NaN,NaN,Gyovani Yuri Souza Santos
1,Jessica Viana,7 de agosto de 2025,NaN,atualizado,v.jessi.carvalho@gmail.com,1 de fevereiro de 2024,Jéssica Carvalho Viana,79999103379,Site do Processo Seletivo LAWD (https://www.no...,NaN,...,79999103379,7 de agosto de 2025,Eventos,"Backend, Design, FrontEnd",atualizado,NaN,Site do Processo Seletivo LAWD (https://www.no...,NaN,NaN,Jéssica Carvalho Viana
2,Maria Luiza,22 de julho de 2025,NaN,atualizado,malufmatos22@gmail.com,5 de fevereiro de 2024,Maria Luiza Ferreira Matos,79981452229,Site do Processo Seletivo LAWD (https://www.no...,NaN,...,79981452229,22 de julho de 2025,"Marketing, Pesquisa, Projetos","Backend, Design, FrontEnd",atualizado,NaN,Site do Processo Seletivo LAWD (https://www.no...,NaN,NaN,Maria Luiza Ferreira Matos


In [26]:
# Tratamento do dataframe do Notion Geral
df_notion = tratar_dataframe_notion(
    df_notion,
    colunas_para_remover=["Resumo da IA", "Seleção", "Arquivos e mídia", "Cadastro"],
    colunas_com_links=["Projetos"],
)

df_notion.head(3)

,Name,Aniversário,Email,Entrada,Nome Completo,Phone,Projetos,Status,Techs,Áreas de atuação,Áreas de interesse
0,Gyovani Santos,19 de março de 2025,gyovani.santos@dcomp.ufs.br,NaN,Gyovani Yuri Souza Santos,79988697393,"WebGarden, FORPROEX, Pseudo-Holograma",membro efetivo,"CAD, Redes",Eventos,"Desenvolvimento web, Redes"
1,Jessica Viana,7 de agosto de 2025,v.jessi.carvalho@gmail.com,1 de fevereiro de 2024,Jéssica Carvalho Viana,79999103379,"Site do Processo Seletivo LAWD, WebGarden",membro efetivo,"Backend, Design, FrontEnd",Ensino,Eventos
2,Maria Luiza,22 de julho de 2025,malufmatos22@gmail.com,5 de fevereiro de 2024,Maria Luiza Ferreira Matos,79981452229,"Site do Processo Seletivo LAWD, FORPROEX, Divi...",membro efetivo,"Backend, Design, FrontEnd",Projetos,"Marketing, Pesquisa, Projetos"


In [27]:
# Leitura dos CSVs do Notion da Diretoria de Projetos
projetos_dir = os.path.join("data", "notion-projetos")

df_projetos1 = pd.read_csv(
    os.path.join(projetos_dir, "Projetos 1dfb762c7e8b80fa84dfc9ad8e619d61_all.csv")
)

df_projetos2 = pd.read_csv(
    os.path.join(projetos_dir, "Projetos 1dfb762c7e8b80fa84dfc9ad8e619d61.csv")
)

# Fazendo o merge dos dois dataframes
df_projetos = pd.merge(df_projetos1, df_projetos2, on="Nome do Projeto")

df_projetos.head(3)

,Nome do Projeto,Data de Encerramento_x,Data de Início_x,Documentação_x,Link para o repositório_x,Prioridade_x,Progresso_x,Responsável_x,Status_x,Time_x,Responsável_y,Status_y,Data de Início_y,Data de Encerramento_y,Prioridade_y,Time_y,Progresso_y,Documentação_y,Link para o repositório_y
0,Fishbase,09/06/2025,01/04/2025,https://docs.google.com/document/d/1j2XSrwcjUu...,NaN,Não se aplica,NaN,Daniel José,Finalizado,"Alípio Fernandes, Daniel José, Davi Bittencour...",Daniel José,Finalizado,01/04/2025,09/06/2025,Não se aplica,"Alípio Fernandes, Daniel José, Davi Bittencour...",NaN,https://docs.google.com/document/d/1j2XSrwcjUu...,NaN
1,UFS.BR,01/12/2026,09/06/2025,https://docs.google.com/document/d/1vb2pt45JFQ...,NaN,Alta,NaN,Gyovani Yuri,Em andamento,"Davi Bittencourt, Gabriel Almeida, Gabriel Arg...",Gyovani Yuri,Em andamento,09/06/2025,01/12/2026,Alta,"Davi Bittencourt, Gabriel Almeida, Gabriel Arg...",NaN,https://docs.google.com/document/d/1vb2pt45JFQ...,NaN
2,Site do PS,15/03/2025,08/05/2024,NaN,NaN,Não se aplica,NaN,Davi Bittencourt,Finalizado,"Davi Bittencourt, Guilherme Linard, Jéssica Vi...",Davi Bittencourt,Finalizado,08/05/2024,15/03/2025,Não se aplica,"Davi Bittencourt, Guilherme Linard, Jéssica Vi...",NaN,NaN,NaN


## Tratamento dos dados

In [28]:
# Tratamento do dataframe do Notion da Diretoria de Projetos
df_projetos = tratar_dataframe_notion(
    df_projetos,
    colunas_para_remover=["Documentação", "Link para o repositório", "Progresso"],
    colunas_com_links=df_projetos.select_dtypes(include=["object", "string"]).columns,
)

df_projetos

,Nome do Projeto,Data de Encerramento,Data de Início,Prioridade,Responsável,Status,Time
0,Fishbase,09/06/2025,01/04/2025,Não se aplica,Daniel José,Finalizado,"Alípio Fernandes, Daniel José, Davi Bittencour..."
1,UFS.BR,01/12/2026,09/06/2025,Alta,Gyovani Yuri,Em andamento,"Davi Bittencourt, Gabriel Almeida, Gabriel Arg..."
2,Site do PS,15/03/2025,08/05/2024,Não se aplica,Davi Bittencourt,Finalizado,"Davi Bittencourt, Guilherme Linard, Jéssica Vi..."
3,Chatbot,03/02/2025,15/04/2024,Não se aplica,João Felipe,Finalizado,"Daniel José, Gabriel Almeida, João Felipe, Álv..."
4,WebGarden,01/08/2025,24/01/2025,Não se aplica,Alípio Fernandes,Suspenso,Alípio Fernandes
5,Site do Calicomp,NaN,NaN,NaN,NaN,Não iniciado,NaN
6,Site do DComp,31/07/2025,27/03/2023,Não se aplica,Daniel José,Suspenso,"Daniel José, Vinicius Morais"
7,Divided Space,NaN,15/03/2024,Não se aplica,Vinícius Morais,Suspenso,NaN
8,Site do LEPFS,18/08/2025,21/07/2025,Não se aplica,Vinícius Morais,Finalizado,"André Felipe, Daniel José, Enzo Emanuel, Lauan..."
9,POA - MPSE,NaN,NaN,Alta,João Felipe,Em andamento,João Felipe


In [29]:
# Alterando os nomes das colunas
df_map = df_map.rename(columns={"nome": "Nome Completo"})

df_notion = df_notion.drop(columns=["Name", "Phone", "Email"])

# Mantém a coluna original como máscara e cria versão sanitizada para matching.
df_map["Nome Completo Original"] = df_map["Nome Completo"]
df_notion["Nome Completo Original"] = df_notion["Nome Completo"]

df_map["Nome Completo"] = df_map["Nome Completo"].apply(sanitize_string)
df_notion["Nome Completo"] = df_notion["Nome Completo"].apply(sanitize_string)

display(df_map.head(3))
display(df_notion.head(3))

display(df_map.count())
display(df_notion.count())

,Nome Completo,curso,periodo,disponibilidade,tech,carreira,maturidade,projetosAtuais,projetosFuturos,Nome Completo Original
0,BRUNO SAINT CLAIR SILVA OLIVEIRA,SI (Sistemas de Informação),8,De 6 a 8 horas semanais,"Java, JavaScript, TypeScript, Python, HTML e C...",Desenvolvedor Back-end,Autônomo (Resolve problemas sozinho),SIMGrade,"Site da LAWD, POA - MPSE",Bruno Saint Clair Silva Oliveira
1,ALVARO LUIS SILVA PEIXOTO,CC (Ciência da Computação),10,Menos de 6 horas semanais,"PHP, Javascript, Golang, Typescript, C#, Ban...","Desenvolvedor Back-end, Full Stack",Autônomo (Resolve problemas sozinho),Nenhum,"ufs.br, Site da LAWD, POA - MPSE, TransformAção",Álvaro Luís Silva Peixoto
2,ANDRE FELIPE DE SANTANA CONCEICAO,SI (Sistemas de Informação),7,De 6 a 8 horas semanais,"HTML, CSS, JS, React, NodeJs, NextJS, Git, Git...",Full Stack,Em desenvolvimento (Já executa tarefas com apoio),Nenhum,"ufs.br, Site da LAWD, SIMCalc, POA - MPSE, Tra...",André Felipe de Santana Conceição


,Aniversário,Entrada,Nome Completo,Projetos,Status,Techs,Áreas de atuação,Áreas de interesse,Nome Completo Original
0,19 de março de 2025,NaN,GYOVANI YURI SOUZA SANTOS,"WebGarden, FORPROEX, Pseudo-Holograma",membro efetivo,"CAD, Redes",Eventos,"Desenvolvimento web, Redes",Gyovani Yuri Souza Santos
1,7 de agosto de 2025,1 de fevereiro de 2024,JESSICA CARVALHO VIANA,"Site do Processo Seletivo LAWD, WebGarden",membro efetivo,"Backend, Design, FrontEnd",Ensino,Eventos,Jéssica Carvalho Viana
2,22 de julho de 2025,5 de fevereiro de 2024,MARIA LUIZA FERREIRA MATOS,"Site do Processo Seletivo LAWD, FORPROEX, Divi...",membro efetivo,"Backend, Design, FrontEnd",Projetos,"Marketing, Pesquisa, Projetos",Maria Luiza Ferreira Matos


Nome Completo             31
curso                     31
periodo                   31
disponibilidade           31
tech                      31
carreira                  31
maturidade                31
projetosAtuais            31
projetosFuturos           31
Nome Completo Original    31
dtype: int64

Aniversário               33
Entrada                   19
Nome Completo             33
Projetos                  12
Status                    33
Techs                     32
Áreas de atuação          32
Áreas de interesse        32
Nome Completo Original    33
dtype: int64

In [30]:
# Mantém todas as pessoas do Notion (33), mesmo sem resposta no forms.
df = pd.merge(
    df_notion,
    df_map,
    on="Nome Completo",
    how="left",
)

df = df.rename(columns={"Nome Completo Original_x": "Nome Completo Original"})
df = df.drop(columns=["Nome Completo Original_y", "Projetos"])

df.head(3)

,Aniversário,Entrada,Nome Completo,Status,Techs,Áreas de atuação,Áreas de interesse,Nome Completo Original,curso,periodo,disponibilidade,tech,carreira,maturidade,projetosAtuais,projetosFuturos
0,19 de março de 2025,NaN,GYOVANI YURI SOUZA SANTOS,membro efetivo,"CAD, Redes",Eventos,"Desenvolvimento web, Redes",Gyovani Yuri Souza Santos,SI (Sistemas de Informação),8.0,De 6 a 8 horas semanais,"TS, React, Tailwind, Next, Nest, JS, Python, F...","Full Stack, Gestor de Projeto / Scrum",Mentor (Ajuda outros membros),"ufs.br, SIMGrade","ufs.br, Site da LAWD"
1,7 de agosto de 2025,1 de fevereiro de 2024,JESSICA CARVALHO VIANA,membro efetivo,"Backend, Design, FrontEnd",Ensino,Eventos,Jéssica Carvalho Viana,Design,7.0,Menos de 6 horas semanais,"Frontend, HTML/css, React e flutter","Desenvolvedor Front-end, UI/UX, Gestor de Proj...",Autônomo (Resolve problemas sozinho),"ufs.br, Dcomp Mulheres",ufs.br
2,22 de julho de 2025,5 de fevereiro de 2024,MARIA LUIZA FERREIRA MATOS,membro efetivo,"Backend, Design, FrontEnd",Projetos,"Marketing, Pesquisa, Projetos",Maria Luiza Ferreira Matos,CC (Ciência da Computação),8.0,De 9 a 12 horas semanais,"HTML, CSS, JS, TS, React","Desenvolvedor Front-end, UI/UX, Gestor de Proj...",Autônomo (Resolve problemas sozinho),"ufs.br, Dcomp Mulheres",SIMCalc


In [31]:
# Reordenando as colunas do dataframe df
cols = ["Nome Completo", "Áreas de atuação", "projetosAtuais", "projetosFuturos", "periodo"]

rest = [col for col in df.columns if col not in cols]

df = df[cols + rest]

df.head(3)

,Nome Completo,Áreas de atuação,projetosAtuais,projetosFuturos,periodo,Aniversário,Entrada,Status,Techs,Áreas de interesse,Nome Completo Original,curso,disponibilidade,tech,carreira,maturidade
0,GYOVANI YURI SOUZA SANTOS,Eventos,"ufs.br, SIMGrade","ufs.br, Site da LAWD",8.0,19 de março de 2025,NaN,membro efetivo,"CAD, Redes","Desenvolvimento web, Redes",Gyovani Yuri Souza Santos,SI (Sistemas de Informação),De 6 a 8 horas semanais,"TS, React, Tailwind, Next, Nest, JS, Python, F...","Full Stack, Gestor de Projeto / Scrum",Mentor (Ajuda outros membros)
1,JESSICA CARVALHO VIANA,Ensino,"ufs.br, Dcomp Mulheres",ufs.br,7.0,7 de agosto de 2025,1 de fevereiro de 2024,membro efetivo,"Backend, Design, FrontEnd",Eventos,Jéssica Carvalho Viana,Design,Menos de 6 horas semanais,"Frontend, HTML/css, React e flutter","Desenvolvedor Front-end, UI/UX, Gestor de Proj...",Autônomo (Resolve problemas sozinho)
2,MARIA LUIZA FERREIRA MATOS,Projetos,"ufs.br, Dcomp Mulheres",SIMCalc,8.0,22 de julho de 2025,5 de fevereiro de 2024,membro efetivo,"Backend, Design, FrontEnd","Marketing, Pesquisa, Projetos",Maria Luiza Ferreira Matos,CC (Ciência da Computação),De 9 a 12 horas semanais,"HTML, CSS, JS, TS, React","Desenvolvedor Front-end, UI/UX, Gestor de Proj...",Autônomo (Resolve problemas sozinho)


In [32]:
# Atualizando o dataframe df com os dados presentes no Notion da Diretoria de Projetos

time_col = "times" if "times" in df_projetos.columns else "Time"
nome_lookup = build_nome_lookup(df)

# Padroniza Time para usar as chaves canônicas presentes em df["Nome Completo"].
df_projetos[time_col] = df_projetos[time_col].apply(
    lambda value: [
        nome
        for nome in (
            normalizar_participante(item, nome_lookup)
            for item in split_participantes(value)
        )
        if nome is not None
    ]
)

# Padroniza Responsável para usar as chaves canônicas presentes em df["Nome Completo"].
responsavel_col = "Responsável"
df_projetos[responsavel_col] = df_projetos[responsavel_col].apply(
    lambda value: ", ".join(
        [
            nome
            for nome in (
                normalizar_participante(item, nome_lookup)
                for item in split_participantes(value)
            )
            if nome is not None
        ]
    ) if not pd.isna(value) else value
)

series_projetos = build_project_series(
    df_projetos=df_projetos,
    nome_lookup=nome_lookup,
    project_col="Nome do Projeto",
    time_col=time_col,
    responsavel_col=responsavel_col,
    status_col="Status",
)

# Evita colisões ao reexecutar a célula no mesmo kernel.
for base_col in series_projetos.keys():
    for suffix in ["", "_x", "_y"]:
        df = df.drop(columns=[f"{base_col}{suffix}"], errors="ignore")

# Aplica no df mantendo Nome Completo como chave
for serie in series_projetos.values():
    df = df.merge(serie, on="Nome Completo", how="left")

df = df.rename(columns={"projetosFuturos": "Projetos com Interesse"})

# Preenche listas vazias para quem não tem projeto associado
for col in [
    "Projetos Atuais",
    "Projetos Finalizados",
    "Projetos Atuais Em Andamento",
    "Projetos Coordenados",
]:
    if col not in df.columns:
        df[col] = [[] for _ in range(len(df))]
    else:
        df[col] = df[col].apply(lambda x: x if isinstance(x, list) else [])

# Converte strings com vírgula em listas nos dataframes df e df_projetos
df = convert_comma_strings_to_lists(df)
df_projetos = convert_comma_strings_to_lists(df_projetos)

# Padroniza chave de projetos (caixa alta, sem acento e trim), preservando nomes originais
project_aliases = {
    "ufs.br": "UFS.BR",
    '"Atualização" do Site do PS': "Site do PS 2.0",
}
df_projetos, df = standardize_project_keys(
    df_projetos=df_projetos,
    df=df,
    project_col="Nome do Projeto",
    df_project_columns=[
        "Projetos Atuais",
        "Projetos Atuais Em Andamento",
        "Projetos Finalizados",
        "Projetos Coordenados",
        "Projetos com Interesse",
    ],
    aliases=project_aliases,
    report_unmatched=True,
)

# Removendo colunas sobre projetos que não fazem mais sentido
df = df.drop(columns=["projetosAtuais", "Projetos Atuais"], errors="ignore")

# Reordenando as colunas do dataframe df
periodo_col = "periodo" if "periodo" in df.columns else "Período"
cols = [
    "Nome Completo",
    "Áreas de atuação",
    "Projetos Atuais Em Andamento",
    "Projetos Finalizados",
    "Projetos Coordenados",
    "Projetos com Interesse",
    periodo_col,
]

rest = [col for col in df.columns if col not in cols]
df = df[cols + rest]

display(df.head(3))
display(df_projetos.head(3))

,Nome Completo,Áreas de atuação,Projetos Atuais Em Andamento,Projetos Finalizados,Projetos Coordenados,Projetos com Interesse,periodo,Aniversário,Entrada,Status,Techs,Áreas de interesse,Nome Completo Original,curso,disponibilidade,tech,carreira,maturidade
0,GYOVANI YURI SOUZA SANTOS,Eventos,"[SIMGRADE, UFS.BR]",[FISHBASE],"[SIMGRADE, UFS.BR]","[UFS.BR, SITE DA LAWD]",8.0,19 de março de 2025,NaN,membro efetivo,"[CAD, Redes]","[Desenvolvimento web, Redes]",Gyovani Yuri Souza Santos,SI (Sistemas de Informação),De 6 a 8 horas semanais,"[TS, React, Tailwind, Next, Nest, JS, Python, ...","[Full Stack, Gestor de Projeto / Scrum]",Mentor (Ajuda outros membros)
1,JESSICA CARVALHO VIANA,Ensino,[],[],[],UFS.BR,7.0,7 de agosto de 2025,1 de fevereiro de 2024,membro efetivo,"[Backend, Design, FrontEnd]",Eventos,Jéssica Carvalho Viana,Design,Menos de 6 horas semanais,"[Frontend, HTML/css, React e flutter]","[Desenvolvedor Front-end, UI/UX, Gestor de Pro...",Autônomo (Resolve problemas sozinho)
2,MARIA LUIZA FERREIRA MATOS,Projetos,[UFS.BR],[],[],SIMCALC,8.0,22 de julho de 2025,5 de fevereiro de 2024,membro efetivo,"[Backend, Design, FrontEnd]","[Marketing, Pesquisa, Projetos]",Maria Luiza Ferreira Matos,CC (Ciência da Computação),De 9 a 12 horas semanais,"[HTML, CSS, JS, TS, React]","[Desenvolvedor Front-end, UI/UX, Gestor de Pro...",Autônomo (Resolve problemas sozinho)


,Nome do Projeto,Data de Encerramento,Data de Início,Prioridade,Responsável,Status,Time,Nome do Projeto Original
0,FISHBASE,09/06/2025,01/04/2025,Não se aplica,DANIEL JOSE SILVA TRINDADE,Finalizado,"[DANIEL JOSE SILVA TRINDADE, DAVI BITTENCOURT ...",Fishbase
1,UFS.BR,01/12/2026,09/06/2025,Alta,GYOVANI YURI SOUZA SANTOS,Em andamento,"[DAVI BITTENCOURT DE ALMEIDA, GABRIEL ARGOLO J...",UFS.BR
2,SITE DO PS,15/03/2025,08/05/2024,Não se aplica,DAVI BITTENCOURT DE ALMEIDA,Finalizado,"[DAVI BITTENCOURT DE ALMEIDA, GUILHERME LINARD...",Site do PS


In [33]:
# Buscar todos os projetos presentes no Dataframe df para padronizar os nomes dos projetos nos dois Dataframes
colunas_candidatas = [
    "Projetos Atuais Em Andamento",
    "Projetos Atuais em Andamento",
    "Projetos Finalizados",
    "Projetos Coordenados",
    "Projetos com Interesse",
]

colunas = [col for col in colunas_candidatas if col in df.columns]
projetos = []
vistos = set()

for col in colunas:
    for valor in df[col].dropna():
        if isinstance(valor, list):
            itens = valor
        elif isinstance(valor, str):
            itens = [item.strip() for item in valor.split(",") if item.strip()]
        else:
            continue
        for item in itens:
            if isinstance(item, str) and item.strip() and item not in vistos:
                vistos.add(item)
                projetos.append(item)

display(projetos)
print(len(projetos))
display(df_projetos['Nome do Projeto'])

['SIMGRADE',
 'UFS.BR',
 'POA - MPSE',
 'CRIAJUNTO',
 'FISHBASE',
 'CHATBOT',
 'SITE DO PS',
 'SITE DO LEPFS',
 'DIVIDED SPACE',
 'SITE DO DCOMP',
 'SIMCALC',
 'TRANSFORMACAO',
 'SITE DA LAWD',
 'LANDING PAGE DA WWWEEK']

14


0                   FISHBASE
1                     UFS.BR
2                 SITE DO PS
3                    CHATBOT
4                  WEBGARDEN
5           SITE DO CALICOMP
6              SITE DO DCOMP
7              DIVIDED SPACE
8              SITE DO LEPFS
9                 POA - MPSE
10                  SIMGRADE
11              SITE DA LAWD
12                   SIMCALC
13             TRANSFORMACAO
14            SITE DO PS 2.0
15    LANDING PAGE DA WWWEEK
16                 CRIAJUNTO
Name: Nome do Projeto, dtype: str

In [34]:
# Unindo os valores da coluna "Techs" com "tech".
df = merge_dataframe_columns_values(
    df=df,
    left_col="Techs",
    right_col="tech",
    target_col="Techs",
    drop_source_columns=True,
)

In [35]:
# Renomeando as colunas de df
df = df.rename(columns={
    'periodo': 'Período',
    'curso': 'Curso',
    'disponibilidade': 'Disponibilidade',
    'carreira': 'Carreira',
    'maturidade': 'Maturidade'
})

# Organizando as colunas de df
cols = [
    "Nome Completo",
    "Áreas de atuação",
    "Projetos Atuais Em Andamento",
    "Projetos Finalizados",
    "Projetos Coordenados",
    "Projetos com Interesse",
    "Techs",
    "Período",
    "Áreas de interesse",
    "Disponibilidade",
    "Carreira",
    "Maturidade",
    "Curso",
    "Entrada",
    "Nome Completo Original"
]

rest = [col for col in df.columns if col not in cols]
df = df[cols + rest]

df.columns

Index(['Nome Completo', 'Áreas de atuação', 'Projetos Atuais Em Andamento',
       'Projetos Finalizados', 'Projetos Coordenados',
       'Projetos com Interesse', 'Techs', 'Período', 'Áreas de interesse',
       'Disponibilidade', 'Carreira', 'Maturidade', 'Curso', 'Entrada',
       'Nome Completo Original', 'Aniversário', 'Status'],
      dtype='str')

In [36]:
# Organizando as colunas do df_projetos
cols = [
    "Nome do Projeto",
    "Responsável",
    "Data de Início",
    "Data de Encerramento",
    "Status",
    "Prioridade",
    "Time",
]

rest = [col for col in df_projetos.columns if col not in cols]
df_projetos = df_projetos[cols + rest]

df_projetos

,Nome do Projeto,Responsável,Data de Início,Data de Encerramento,Status,Prioridade,Time,Nome do Projeto Original
0,FISHBASE,DANIEL JOSE SILVA TRINDADE,01/04/2025,09/06/2025,Finalizado,Não se aplica,"[DANIEL JOSE SILVA TRINDADE, DAVI BITTENCOURT ...",Fishbase
1,UFS.BR,GYOVANI YURI SOUZA SANTOS,09/06/2025,01/12/2026,Em andamento,Alta,"[DAVI BITTENCOURT DE ALMEIDA, GABRIEL ARGOLO J...",UFS.BR
2,SITE DO PS,DAVI BITTENCOURT DE ALMEIDA,08/05/2024,15/03/2025,Finalizado,Não se aplica,"[DAVI BITTENCOURT DE ALMEIDA, GUILHERME LINARD...",Site do PS
3,CHATBOT,JOAO FELIPE QUENTINO,15/04/2024,03/02/2025,Finalizado,Não se aplica,"[DANIEL JOSE SILVA TRINDADE, JOAO FELIPE QUENT...",Chatbot
4,WEBGARDEN,,24/01/2025,01/08/2025,Suspenso,Não se aplica,[],WebGarden
5,SITE DO CALICOMP,NaN,NaN,NaN,Não iniciado,NaN,[],Site do Calicomp
6,SITE DO DCOMP,DANIEL JOSE SILVA TRINDADE,27/03/2023,31/07/2025,Suspenso,Não se aplica,"[DANIEL JOSE SILVA TRINDADE, VINICIUS MORAIS S...",Site do DComp
7,DIVIDED SPACE,VINICIUS MORAIS SOUZA,15/03/2024,NaN,Suspenso,Não se aplica,[],Divided Space
8,SITE DO LEPFS,VINICIUS MORAIS SOUZA,21/07/2025,18/08/2025,Finalizado,Não se aplica,"[ANDRE FELIPE DE SANTANA CONCEICAO, DANIEL JOS...",Site do LEPFS
9,POA - MPSE,JOAO FELIPE QUENTINO,NaN,NaN,Em andamento,Alta,[JOAO FELIPE QUENTINO],POA - MPSE


In [37]:
# Adicionando os nomes dos Coordenadores de Projeto em df_projetos
valid_keys = set(df["Nome Completo"].dropna())
responsaveis = set()
for valor in df_projetos["Responsável"].dropna():
    for item in split_participantes(valor):
        if item:
            responsaveis.add(item)
sorted(responsaveis - valid_keys)[:20], len(responsaveis - valid_keys)

([], 0)

In [38]:
# Remova os projetos que estão em "Projetos Atuais em Andamento" de "Projetos com Interesse"
df["Projetos com Interesse"] = df.apply(
    lambda row: [
        p for p in (row["Projetos com Interesse"] if isinstance(row["Projetos com Interesse"], list) else [])
        if p not in (row["Projetos Atuais Em Andamento"] if isinstance(row["Projetos Atuais Em Andamento"], list) else [])
    ],
    axis=1
)

# Converte "Período" para inteiro (nullable) de forma segura
df["Período"] = pd.to_numeric(df["Período"], errors="coerce").astype("Int64")

In [39]:
df_projetos.head(5)

,Nome do Projeto,Responsável,Data de Início,Data de Encerramento,Status,Prioridade,Time,Nome do Projeto Original
0,FISHBASE,DANIEL JOSE SILVA TRINDADE,01/04/2025,09/06/2025,Finalizado,Não se aplica,"[DANIEL JOSE SILVA TRINDADE, DAVI BITTENCOURT ...",Fishbase
1,UFS.BR,GYOVANI YURI SOUZA SANTOS,09/06/2025,01/12/2026,Em andamento,Alta,"[DAVI BITTENCOURT DE ALMEIDA, GABRIEL ARGOLO J...",UFS.BR
2,SITE DO PS,DAVI BITTENCOURT DE ALMEIDA,08/05/2024,15/03/2025,Finalizado,Não se aplica,"[DAVI BITTENCOURT DE ALMEIDA, GUILHERME LINARD...",Site do PS
3,CHATBOT,JOAO FELIPE QUENTINO,15/04/2024,03/02/2025,Finalizado,Não se aplica,"[DANIEL JOSE SILVA TRINDADE, JOAO FELIPE QUENT...",Chatbot
4,WEBGARDEN,,24/01/2025,01/08/2025,Suspenso,Não se aplica,[],WebGarden


In [40]:
df.head(5)

,Nome Completo,Áreas de atuação,Projetos Atuais Em Andamento,Projetos Finalizados,Projetos Coordenados,Projetos com Interesse,Techs,Período,Áreas de interesse,Disponibilidade,Carreira,Maturidade,Curso,Entrada,Nome Completo Original,Aniversário,Status
0,GYOVANI YURI SOUZA SANTOS,Eventos,"[SIMGRADE, UFS.BR]",[FISHBASE],"[SIMGRADE, UFS.BR]",[SITE DA LAWD],"[CAD, Redes, TS, React, Tailwind, Next, Nest, ...",8,"[Desenvolvimento web, Redes]",De 6 a 8 horas semanais,"[Full Stack, Gestor de Projeto / Scrum]",Mentor (Ajuda outros membros),SI (Sistemas de Informação),NaN,Gyovani Yuri Souza Santos,19 de março de 2025,membro efetivo
1,JESSICA CARVALHO VIANA,Ensino,[],[],[],[],"[Backend, Design, FrontEnd, Frontend, HTML/css...",7,Eventos,Menos de 6 horas semanais,"[Desenvolvedor Front-end, UI/UX, Gestor de Pro...",Autônomo (Resolve problemas sozinho),Design,1 de fevereiro de 2024,Jéssica Carvalho Viana,7 de agosto de 2025,membro efetivo
2,MARIA LUIZA FERREIRA MATOS,Projetos,[UFS.BR],[],[],[],"[Backend, Design, FrontEnd, HTML, CSS, JS, TS,...",8,"[Marketing, Pesquisa, Projetos]",De 9 a 12 horas semanais,"[Desenvolvedor Front-end, UI/UX, Gestor de Pro...",Autônomo (Resolve problemas sozinho),CC (Ciência da Computação),5 de fevereiro de 2024,Maria Luiza Ferreira Matos,22 de julho de 2025,membro efetivo
3,JOAO FELIPE QUENTINO,Ensino,"[POA - MPSE, UFS.BR]",[CHATBOT],"[CHATBOT, POA - MPSE]",[],"[Automação, IA, Machine Learning, Python, pyth...",9,"[IA, Pesquisa]",Menos de 6 horas semanais,"[Desenvolvedor Back-end, Pesquisador / Documen...",Mentor (Ajuda outros membros),SI (Sistemas de Informação),NaN,João Felipe Quentino,6 de abril de 2025,membro efetivo
4,WANESSA SILVA SANTOS,RH,[],[],[],[],"[Adm, Java, Vanilla, Python, Javascript e C.]",8,"[Projetos, RH]",De 6 a 8 horas semanais,"[Desenvolvedor Front-end, Gestor de Projeto / ...",Autônomo (Resolve problemas sozinho),SI (Sistemas de Informação),NaN,Wanessa Silva Santos,27 de abril de 2025,diretor


## Exportando os dados para um JSON

In [ ]:
# Criando o JSON
from pathlib import Path

db = {
    "membros": df.to_dict(orient="records"),
    "projetos": df_projetos.to_dict(orient="records")
}

db = replace_nan_with_none(db)

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

output_path = project_root / "db" / "db.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(db, f, ensure_ascii=False, indent=2)

print(f"db.json salvo em: {output_path}")